# Raw data research: Intent AI simulation

<span style="background:#c2352b;color:#fff;padding:4px 12px;border-radius:99px;font-weight:700">SOURCE: TEST</span>

All data in this notebook comes from the local **TEST DB** (`presentation/simulation/data/test.db`), not production. Every table has a `SOURCE` column with the value `TEST`.

> ⚠️ **Raw personal data:** transcripts, recordings and anonymous identifiers of end users. The connection is **read-only**. **Before a commit, run `Clear All Outputs`** so that personal data doesn't end up in git.

**How to run:** select a Python kernel (top right), then `Run All`.

In [ ]:
# --- Setup: read-only connection to the TEST DB ---
import json
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import Audio, HTML, display

SOURCE = "TEST"
ROW_LIMIT = 5000

# The notebook sits in presentation/simulation/notebooks/ → look upward for the simulation folder
SIM_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sites.json").exists() and (p / "server.js").exists()),
               Path.cwd().parent)
DB_PATH = SIM_DIR / "data" / "test.db"
AUDIO_DIR = SIM_DIR / "data" / "audio"
assert DB_PATH.exists(), f"TEST DB not found: {DB_PATH} (run npm start in presentation/simulation/ at least once)"

# mode=ro → SQLite blocks any write at the engine level, not just by text filtering
# isolation_level=None → autocommit: no transaction is held open, so the server is never blocked
conn = sqlite3.connect(f"{DB_PATH.as_uri()}?mode=ro", uri=True, isolation_level=None)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


def sql(query: str, params: tuple = ()) -> pd.DataFrame:
    """Runs a read query against the TEST DB and returns a DataFrame with SOURCE=TEST as the first column."""
    df = pd.read_sql_query(query, conn, params=params)
    if len(df) > ROW_LIMIT:
        print(f"Showing the first {ROW_LIMIT} of {len(df)} rows")
        df = df.head(ROW_LIMIT)
    df.insert(0, "SOURCE", SOURCE)
    return df


display(HTML(f'<b style="color:#c2352b">SOURCE: {SOURCE}</b> · {DB_PATH} · read-only'))

In [ ]:
# Check: the connection really is read-only (a write attempt should be blocked)
try:
    conn.execute("DELETE FROM feedback WHERE 1 = 0")
    print("⚠️  Warning: the connection is not read-only!")
except sqlite3.OperationalError as e:
    print(f"✅ Read-only: {e}")

## 1. Overview by site (customer)
`active = 0` means a site that was removed from `sites.json`. Its data is kept in the DB but hidden from the dashboard.

In [ ]:
sql("""
SELECT o.slug, o.name, o.industry, o.active,
       COUNT(f.id)                              AS total,
       SUM(f.type = 'voice')                    AS voice,
       SUM(f.type = 'text')                     AS text,
       COUNT(DISTINCT f.end_user_id)            AS end_users,
       MIN(f.created_at)                        AS first_feedback,
       MAX(f.created_at)                        AS last_feedback
FROM organizations o
JOIN products p      ON p.organization_id = o.id
LEFT JOIN feedback f ON f.product_id = p.id
GROUP BY o.id
ORDER BY total DESC
""")

## 2. All raw data, record by record
Every field as it's stored in the DB, joined with the site and the end user. The `raw` table is available for further analysis in pandas.

In [ ]:
# Newer fields may be missing from an older DB (the server adds them on start; this connection is read-only)
_fb_cols = {r[1] for r in conn.execute("PRAGMA table_info(feedback)")}
_optional = ["consent_version", "consent_at", "browser", "os", "device", "language", "time_zone", "country", "ip_hash", "feedback_lang", "channel"]
_opt_sql = ", ".join(f"f.{c}" if c in _fb_cols else f"NULL AS {c}" for c in _optional)
_missing = [c for c in _optional if c not in _fb_cols]
if _missing:
    print(f"Note: these fields don't exist yet in the DB (restart the server): {', '.join(_missing)}")

raw = sql(f"""
SELECT f.id, o.slug AS site, o.name AS site_name, o.active AS site_active,
       f.type, f.content, f.rating, f.sentiment, f.category,
       f.audio_path, f.audio_duration_ms, f.page_url, f.created_at,
       {_opt_sql},
       e.external_id AS end_user, e.first_seen AS end_user_first_seen, e.last_seen AS end_user_last_seen,
       p.id AS product_id, f.end_user_id AS end_user_row_id
FROM feedback f
JOIN products p      ON p.id = f.product_id
JOIN organizations o ON o.id = p.organization_id
JOIN end_users e     ON e.id = f.end_user_id
ORDER BY f.created_at DESC
""")
# Optional: exclude automated test rows (content starting with "QA-") from the analysis
EXCLUDE_QA = False
if EXCLUDE_QA:
    raw = raw[~raw["content"].fillna("").str.startswith("QA-")].reset_index(drop=True)

print(f"{len(raw)} records · {raw['end_user'].nunique()} distinct users · SOURCE={SOURCE}")
raw.head(20)

## 3. A single record
`record(n)` shows record number `n` from the `raw` table (0 = the newest), or a record by its `id`. It displays all the raw fields, a player if there's a recording, and the record as JSON.

In [ ]:
def record(key=0):
    """Shows a single record by its position in raw (int) or by its feedback id (str)."""
    if isinstance(key, int):
        if not 0 <= key < len(raw):
            raise IndexError(f"There are {len(raw)} records (0–{len(raw) - 1})")
        row = raw.iloc[key]
    else:
        match = raw[raw["id"] == key]
        if match.empty:
            raise KeyError(f"Record {key} not found")
        row = match.iloc[0]

    display(HTML(f'<h4 style="margin:4px 0">Record {row["id"]} '
                 f'<span style="background:#c2352b;color:#fff;padding:2px 10px;border-radius:99px;font-size:12px">SOURCE: {row["SOURCE"]}</span></h4>'))
    display(row.to_frame("value"))

    audio_path = row.get("audio_path")
    if isinstance(audio_path, str) and audio_path:
        f = AUDIO_DIR / Path(audio_path).name
        if f.exists():
            display(Audio(filename=str(f)))
        else:
            print(f"(The recording file no longer exists: {f.name}, possibly deleted by retention or a deletion request)")

    print(json.dumps({k: (None if pd.isna(v) else v) for k, v in row.to_dict().items()}, ensure_ascii=False, indent=2, default=str))


record(0)  # the newest record. Try: record(3) or record("<id>")

## 4. Exploration

In [ ]:
# Filter by site: replace "shufersal" with elal / ynet / superpharm / partner
SITE = "shufersal"
raw[raw["site"] == SITE][["SOURCE", "created_at", "type", "content", "rating", "sentiment", "category"]]

In [ ]:
# Voice feedback without a transcript (the browser's transcription didn't return text)
raw[(raw["type"] == "voice") & (raw["content"].isna())][["SOURCE", "site", "created_at", "audio_path", "audio_duration_ms"]]

In [ ]:
# Sentiment distribution by site
sentiment = pd.crosstab(raw["site"], raw["sentiment"], margins=True, margins_name="Total")
sentiment.insert(0, "SOURCE", SOURCE)
sentiment

In [ ]:
# Category distribution by site
categories = pd.crosstab(raw["category"], raw["site"])
categories.insert(0, "SOURCE", SOURCE)
categories

In [ ]:
# Activity over time (feedback per day)
daily = raw.assign(day=pd.to_datetime(raw["created_at"]).dt.date).groupby(["day", "site"]).size().unstack(fill_value=0)
daily.insert(0, "SOURCE", SOURCE)
daily

In [ ]:
# Free SQL query: write any read query (writes are blocked)
sql("""
SELECT o.name AS site, f.category, COUNT(*) AS n, ROUND(AVG(f.rating), 2) AS avg_rating
FROM feedback f
JOIN products p      ON p.id = f.product_id
JOIN organizations o ON o.id = p.organization_id
GROUP BY o.name, f.category
ORDER BY n DESC
""")

## 5. Trend charts
Every chart comes with a data table underneath (accessibility, and exact numbers). Colors follow the site (a fixed order from `sites.json`), not its rank. Hours are shown in Israel time. Hidden sites (`site_active = 0`) are excluded from the charts.

In [ ]:
# --- Chart style (reference palette from the dataviz guidelines, validated for color blindness) ---
import re
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from matplotlib.ticker import MaxNLocator

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQ = mcolors.LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#2a78d6", "#1c5cab", "#0d366b"])
NEG, NEU, POS = "#e34948", "#f0efec", "#2a78d6"   # diverging: red ↔ gray ↔ blue
TZ = "Asia/Jerusalem"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": ["Segoe UI", "Arial", "DejaVu Sans"], "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "right",
})

# Hebrew in charts: matplotlib 3.11+ lays out right-to-left text on its own (an extra conversion would mirror it);
# older versions need bidi reordering. Pick the right method automatically.
_mpl = tuple(int(x) for x in matplotlib.__version__.split(".")[:2])
if _mpl >= (3, 11):
    h = str
else:
    try:
        from bidi import get_display as h
    except ImportError:
        h = lambda t: str(t)[::-1]  # rough fallback for pure Hebrew labels
print(f"matplotlib {matplotlib.__version__} · Hebrew in charts: {'native' if h is str else 'bidi conversion'}")

SITE_ORDER = [s["slug"] for s in json.loads((SIM_DIR / "sites.json").read_text(encoding="utf-8"))]
SITE_COLOR = {slug: SERIES[i] for i, slug in enumerate(SITE_ORDER)}
SITE_NAME = dict(zip(raw["site"], raw["site_name"]))

active = raw[raw["site_active"] == 1].copy()
active["local_time"] = pd.to_datetime(active["created_at"], utc=True).dt.tz_convert(TZ)
active["hour"] = active["local_time"].dt.hour
active["day"] = active["local_time"].dt.date
sites_present = [s for s in SITE_ORDER if s in set(active["site"])]


def source_tag(ax):
    ax.text(0, 1.02, f"SOURCE: {SOURCE}", transform=ax.transAxes, ha="left", va="bottom",
            fontsize=8, color="#c2352b", fontweight="bold")


def table(df):
    out = df.copy()
    out.insert(0, "SOURCE", SOURCE)
    display(out)


print(f"{len(active)} records from {len(sites_present)} active sites · SOURCE={SOURCE}")

### 5.1 Hours of the day: when do users send feedback?

In [ ]:
by_hour = active.groupby("hour").size().reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(by_hour.index, by_hour.values, width=0.8, color=SERIES[0], edgecolor=SURFACE, linewidth=2)
ax.set_xticks(range(0, 24, 2), [f"{h_:02d}:00" for h_ in range(0, 24, 2)])
ax.set_ylabel(h("משובים")); ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title(h("משובים לפי שעה ביממה (שעון ישראל)"))
ax.grid(axis="x", visible=False)
peak = int(by_hour.idxmax())
ax.annotate(h(f"שיא: {by_hour.max()}"), (peak, by_hour.max()), xytext=(0, 4), textcoords="offset points",
            ha="center", fontsize=9, color=INK)
source_tag(ax)
plt.tight_layout(); plt.show()
table(by_hour.rename("feedback").to_frame().T)

In [ ]:
# Hour × site: where the load concentrates (sequential scale, single hue: light = few, dark = many)
heat = active.pivot_table(index="site", columns="hour", values="id", aggfunc="count", fill_value=0)     .reindex(index=sites_present, columns=range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 0.55 * len(heat) + 1.4))
cmap = SEQ.copy(); cmap.set_bad(SURFACE)  # zero cells = background, not "a little"
im = ax.imshow(np.ma.masked_equal(heat.values, 0), aspect="auto", cmap=cmap, vmin=0.5)
ax.set_xticks(np.arange(-0.5, 24, 1), minor=True); ax.set_yticks(np.arange(-0.5, len(heat), 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2); ax.tick_params(which="minor", length=0)
ax.set_yticks(range(len(heat)), [h(SITE_NAME.get(s, s)) for s in heat.index])
ax.set_xticks(range(0, 24, 2), [f"{x:02d}" for x in range(0, 24, 2)])
ax.set_xlabel(h("שעה"))
ax.grid(which="major", visible=False)
for spine in ax.spines.values(): spine.set_visible(False)
ax.set_title(h("מפת חום: שעה × אתר"))
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.outline.set_visible(False); cb.set_label(h("משובים")); cb.locator = MaxNLocator(integer=True); cb.update_ticks()
source_tag(ax)
plt.tight_layout(); plt.show()
table(heat.rename(index=SITE_NAME))

### 5.2 Sites: volume, distinct users, and trend over time

In [ ]:
per_site = active.groupby("site").agg(feedback=("id", "count"), distinct_users=("end_user", "nunique"))     .reindex(sites_present).fillna(0).astype(int)
per_site["feedback_per_user"] = (per_site["feedback"] / per_site["distinct_users"].where(per_site["distinct_users"] > 0)).round(2)

# Two separate charts (two measures → two charts, no dual axis)
fig, axes = plt.subplots(1, 2, figsize=(11, 0.5 * len(per_site) + 1.6), sharey=True)
labels = [h(SITE_NAME.get(s, s)) for s in per_site.index]
colors = [SITE_COLOR[s] for s in per_site.index]
for ax, col, title in [(axes[0], "feedback", "משובים לפי אתר"), (axes[1], "distinct_users", "משתמשים ייחודיים לפי אתר")]:
    bars = ax.barh(labels, per_site[col], color=colors, edgecolor=SURFACE, linewidth=2, height=0.62)
    ax.bar_label(bars, padding=3, fontsize=9, color=INK)
    ax.set_title(h(title)); ax.grid(axis="y", visible=False); ax.invert_yaxis()
    ax.margins(x=0.15); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
source_tag(axes[0])
plt.tight_layout(); plt.show()
table(per_site.rename(index=SITE_NAME))

In [ ]:
# Daily trend per site (line per site; legend + label at the end of each line)
daily = active.groupby(["day", "site"]).size().unstack(fill_value=0).reindex(columns=sites_present, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 3.8))
for s in daily.columns:
    ax.plot(daily.index, daily[s], color=SITE_COLOR[s], linewidth=2, marker="o", markersize=7,
            markeredgecolor=SURFACE, markeredgewidth=2, label=h(SITE_NAME.get(s, s)))
    if len(daily) > 1:
        ax.annotate(h(SITE_NAME.get(s, s)), (daily.index[-1], daily[s].iloc[-1]), xytext=(6, 0),
                    textcoords="offset points", va="center", fontsize=9, color=INK_2)
ax.set_ylabel(h("משובים ביום")); ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title(h("מגמה יומית לפי אתר"))
ax.legend(frameon=False, ncols=min(5, len(daily.columns)), loc="upper left", bbox_to_anchor=(0, -0.12))
ax.grid(axis="x", visible=False)
if len(daily) == 1:
    ax.text(0.5, 0.5, h("כל הנתונים מיום אחד — המגמה תתמלא עם הזמן"), transform=ax.transAxes,
            ha="center", color=INK_2, fontsize=9)
source_tag(ax)
plt.tight_layout(); plt.show()
table(daily.rename(columns=SITE_NAME))

### 5.3 Content: topics, sentiment and frequent words

In [ ]:
CATEGORY_HE = {"checkout": "תשלום וקופה", "travel": "טיסות ונסיעה", "delivery": "משלוחים", "prices": "מחירים",
               "billing": "חיובים וחשבונות", "connectivity": "קליטה ואינטרנט", "pharmacy": "מרשמים ותרופות",
               "content": "תוכן ופרסומות", "site": "האתר", "service": "שירות", "other": "אחר"}
cats = active["category"].fillna("other").value_counts()

fig, ax = plt.subplots(figsize=(9, 0.42 * len(cats) + 1.3))
bars = ax.barh([h(CATEGORY_HE.get(c, c)) for c in cats.index], cats.values, color=SERIES[0],
               edgecolor=SURFACE, linewidth=2, height=0.62)
ax.bar_label(bars, padding=3, fontsize=9, color=INK)
ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.margins(x=0.12); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title(h("נושאי המשוב"))
source_tag(ax)
plt.tight_layout(); plt.show()
table(pd.crosstab(active["category"].fillna("other").map(CATEGORY_HE), active["site"].map(SITE_NAME)))

In [ ]:
# Sentiment by site: 100% stacked bars, diverging (negative ↔ neutral ↔ positive)
sent = pd.crosstab(active["site"], active["sentiment"]).reindex(index=sites_present, columns=["negative", "neutral", "positive"], fill_value=0)
share = sent.div(sent.sum(axis=1).where(lambda x: x > 0), axis=0).fillna(0) * 100

fig, ax = plt.subplots(figsize=(10, 0.55 * len(share) + 1.5))
left = pd.Series(0.0, index=share.index)
labels = [h(SITE_NAME.get(s, s)) for s in share.index]
for col, color, name in [("negative", NEG, "שלילי"), ("neutral", NEU, "ניטרלי"), ("positive", POS, "חיובי")]:
    ax.barh(labels, share[col], left=left, color=color, edgecolor=SURFACE, linewidth=2, height=0.62, label=h(name))
    for i, (v, l0) in enumerate(zip(share[col], left)):
        if v >= 12:
            ax.text(l0 + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=9,
                    color="#ffffff" if col != "neutral" else INK)
    left += share[col]
ax.set_xlim(0, 100); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("%"); ax.set_title(h("התפלגות סנטימנט לפי אתר"))
ax.legend(frameon=False, ncols=3, loc="upper left", bbox_to_anchor=(0, -0.18))
source_tag(ax)
plt.tight_layout(); plt.show()
table(sent.rename(index=SITE_NAME))

In [ ]:
# Most frequent words in the content (without stop words, QA identifiers and numbers)
STOP = set("של את על עם זה זו לא כן גם אבל או אם כי הוא היא הם הן אני אנחנו אתם יש אין היה היתה היו להיו מאוד ממש רק עוד כל מה איך למה כמו אחרי לפני תוך בלי כבר שלי שלכם האם".split())
words = Counter(
    w for text in active["content"].dropna()
    for w in re.findall(r"[֐-׿A-Za-z]{3,}", re.sub(r"QA-\S+", " ", text))
    if w not in STOP and not w.startswith("QA")
)
top = pd.Series(dict(words.most_common(15)), dtype=int)

if top.empty:
    print("Not enough text to analyze yet")
else:
    fig, ax = plt.subplots(figsize=(9, 0.38 * len(top) + 1.3))
    bars = ax.barh([h(w) for w in top.index], top.values, color=SERIES[0], edgecolor=SURFACE, linewidth=2, height=0.62)
    ax.bar_label(bars, padding=3, fontsize=9, color=INK)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.margins(x=0.12); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title(h("המילים הנפוצות במשובים"))
    source_tag(ax)
    plt.tight_layout(); plt.show()
    table(top.rename("count").to_frame().T)

### 5.4 Who are the users: distinct users and device context

In [ ]:
ctx = active.assign(
    browser=active["browser"].fillna("לא ידוע (לפני האיסוף)"),
    device=active["device"].fillna("לא ידוע"),
    country=active["country"].fillna("לא ידוע"),
)
users = ctx.groupby("end_user").agg(
    feedback=("id", "count"), sites=("site", "nunique"), first=("created_at", "min"), last=("created_at", "max"),
    browser=("browser", "last"), os=("os", "last"), device=("device", "last"), country=("country", "last"),
    ip_hash=("ip_hash", "last"),
).sort_values("feedback", ascending=False)
print(f"{len(users)} distinct users · {len(active)} feedback items · average {len(active) / max(len(users), 1):.1f} per user")

browser_users = ctx.groupby("browser")["end_user"].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 0.42 * len(browser_users) + 1.3))
bars = ax.barh([h(b) for b in browser_users.index], browser_users.values, color=SERIES[0],
               edgecolor=SURFACE, linewidth=2, height=0.62)
ax.bar_label(bars, padding=3, fontsize=9, color=INK)
ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.margins(x=0.12); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title(h("משתמשים ייחודיים לפי דפדפן"))
source_tag(ax)
plt.tight_layout(); plt.show()
table(users.head(25))

## 6. Data dictionary
| Field | Meaning |
|---|---|
| `SOURCE` | Where the data comes from. Always `TEST` here (the local TEST DB) |
| `type` | `text` / `voice` |
| `content` | The feedback text, or the transcript of a recording (empty if there was no transcript) |
| `rating` | 1–5, text feedback only (optional) |
| `sentiment` / `category` | The result of the local keyword-based analysis (not AI) |
| `audio_path` | The recording file in `data/audio/` |
| `page_url` | The page address without query parameters (data minimization) |
| `consent_version` / `consent_at` | The privacy-notice version the user agreed to, and when |
| `end_user` | Random anonymous identifier of the browser (not a name, email or IP). The same `end_user` = the same author |
| `browser` / `os` / `device` | A summary of the User-Agent only (e.g. Chrome · Windows · desktop); the full string isn't stored |
| `language` / `time_zone` | Browser language and time zone (e.g. `he-IL`, `Asia/Jerusalem`) |
| `country` | Estimated from the time zone, otherwise from the language region (no geolocation) |
| `feedback_lang` | The language the user chose in the widget: `he` / `en` / `ar` |
| `channel` | `shared` = came in through the shared link (another phone or computer), `local` = sent from the demo computer |
| `ip_hash` | One-way HMAC-SHA256 fingerprint of the IP address (the address itself isn't stored). Locally it's always the same value, because everything comes from 127.0.0.1 |
| `site_active` | 0 = a site removed from `sites.json` (its data is kept) |

Retention: rows older than 90 days are deleted automatically by the server.

In [ ]:
conn.close()
print("The connection to the TEST DB is closed.")